# 02b — Find Style + Persona Features

Reuse notebook 02 pipeline with non-emotion prompt sets:
1. **Styles** — biblical, shakespearean, 1920s slang, academic, txt-speak, pirate, noir, gen-z
2. **Personas** — conspiracy theorist, stoic, scientist, valley girl, corporate exec, new age guru, pessimist

Saves `features_styles.json` and `features_personas.json`.

In [1]:
import json
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sae_lens import SAE

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
MODEL_ID = 'google/gemma-3-1b-it'
LAYER = 13
SAE_ID = f'layer_{LAYER}_width_16k_l0_medium'

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map=device).eval()
sae = SAE.from_pretrained(release='gemma-scope-2-1b-it-res', sae_id=SAE_ID, device=device)
print('ready')

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

ready


## Shared utilities

In [2]:
@torch.no_grad()
def feat_activations(prompts, batch_size=8, max_length=80):
    cache = {}
    def hook(_, __, output):
        cache['h'] = (output[0] if isinstance(output, tuple) else output).detach()
    handle = model.model.layers[LAYER].register_forward_hook(hook)
    out = []
    try:
        for i in range(0, len(prompts), batch_size):
            batch = prompts[i:i+batch_size]
            enc = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=max_length).to(device)
            _ = model(**enc)
            h = cache['h'].to(torch.float32)
            feats = sae.encode(h)
            mask = enc['attention_mask'].unsqueeze(-1).float()
            per_prompt = (feats * mask).sum(1) / mask.sum(1).clamp(min=1)
            out.append(per_prompt.cpu())
    finally:
        handle.remove()
    return torch.cat(out, 0)


def discover(prompt_sets, neutral, top_k=15):
    """Run discovery: prompt_sets is dict[label, list[str]]. Returns {label: [{feat_id, diff_score, ...}]}."""
    neutral_acts = feat_activations(neutral)
    neutral_mean = neutral_acts.mean(0)
    neutral_max = neutral_acts.max(0).values
    results = {}
    for label, prompts in prompt_sets.items():
        acts = feat_activations(prompts)
        diff = acts.mean(0) - neutral_mean
        top_idx = torch.topk(diff, top_k).indices.tolist()
        consistency = (acts > neutral_max.unsqueeze(0)).float().mean(0)
        rows = [
            {
                'feat_id': int(idx),
                'diff_score': float(diff[idx]),
                'set_mean_act': float(acts.mean(0)[idx]),
                'neutral_mean_act': float(neutral_mean[idx]),
                'consistency': float(consistency[idx]),
            }
            for idx in top_idx
        ]
        results[label] = rows
        print(f'\n=== {label} ===')
        for r in rows[:6]:
            print(f'  feat {r["feat_id"]:>5d}  diff={r["diff_score"]:.2f}  cons={r["consistency"]:.2f}  neutral={r["neutral_mean_act"]:.2f}')
    return results


NEUTRAL = [
    'The capital of France is Paris.',
    'Water boils at one hundred degrees Celsius.',
    'The library opens at nine in the morning.',
    'I went to the store to buy bread.',
    'The train arrives at platform three.',
    'Photosynthesis converts sunlight into chemical energy.',
    'The meeting is scheduled for Thursday afternoon.',
    'A triangle has three sides and three angles.',
    'The package will arrive next Tuesday.',
    'He picked up the book and turned the page.',
    'Coffee is grown in many tropical countries.',
    'The road continues for another five miles.',
    'My laptop has sixteen gigabytes of memory.',
    'The recipe calls for two cups of flour.',
    'Birds typically migrate south for the winter.',
    'The conference room is on the second floor.',
    'She closed the window because it was raining.',
    'Mathematics is taught in schools worldwide.',
    'The bus stops at the corner every ten minutes.',
    'I read the article this morning.',
]

## 1. Writing styles

8 styles, 10 prompts each. Each prompt is a complete utterance in that style — discovery finds features that fire on the *style* not the *content*.

In [3]:
STYLES = {
    'biblical': [
        'Verily, thus saith the Lord unto His servants.',
        'And lo, a great light appeared in the heavens above.',
        'Behold the wisdom of the prophets and their teachings.',
        'Blessed are the meek, for they shall inherit the earth.',
        'Hearken unto the words of the holy scriptures, O ye people.',
        'Thus it is written, and thus it shall come to pass.',
        'Yea, though I walk through the valley of shadow, I shall fear no evil.',
        'And the Lord spake unto Moses, saying these very words.',
        'O ye of little faith, why do ye doubt the promises of old.',
        'Rejoice, for the kingdom of heaven is at hand.',
    ],
    'shakespearean': [
        'Hark! What light through yonder window breaks so fair?',
        'To be or not to be, that is the eternal question.',
        'Methinks the lady doth protest too much, mine lord.',
        'Forsooth, a pox upon thy treacherous house and kin.',
        'Wherefore art thou, my beloved, in this lonely night?',
        'Anon, good sirrah, prithee tell me thy true name.',
        'Get thee to a nunnery, woman, and trouble me no more.',
        'O villain, villain, smiling damned villain that thou art.',
        'Marry, but the day is fair and the wind is sweet.',
        'I do beseech thee, gentle cousin, lend me thy ear.',
    ],
    '1920s_slang': [
        'The bees knees, doll, this party is the cats pajamas!',
        'What a swell broad, she really takes the cake tonight.',
        'Bee in your bonnet about the boss, see? Stop bellyaching.',
        'That fella is a real Joe Brooks, I tell ya, the real McCoy.',
        'Hot diggity dog, this hooch is mighty fine, baby.',
        'Twentythree skidoo, copper, before I dust ya one.',
        'Aint that the bunk, pal, what a load of horsefeathers.',
        'Oh you kid, what a flapper, dressed to the nines.',
        'Heres mud in your eye, doll, drink up while ya can.',
        'Big cheese rolled up in a swanky jalopy yesterday.',
    ],
    'academic_jargon': [
        'Empirical evidence demonstrates a statistically significant correlation between variables.',
        'The extant literature suggests a paradigmatic shift toward heterogeneous methodologies.',
        'Leveraging a multidisciplinary framework, we operationalize the construct of agency.',
        'Findings indicate non-trivial implications for theoretical and applied domains alike.',
        'Subsequent analysis controlled for confounding sociodemographic covariates.',
        'The hermeneutic approach foregrounds intersubjective meaning-making practices.',
        'A robust regression yielded a coefficient of determination of 0.78.',
        'Notwithstanding methodological limitations, the results are broadly generalizable.',
        'This paper interrogates the epistemological foundations of late-stage capitalism.',
        'We propose a tripartite taxonomy of discursive interventions in the field.',
    ],
    'txt_speak': [
        'omg lol that was sooo cool tbh ngl',
        'fr fr no cap that party was lit af',
        'idk man, ttyl, gotta bounce rn',
        'lmao u for real rn?? smh deadass',
        'wyd later? wanna link up? hmu fam',
        'kk bet, see u in 5, otw rn',
        'wbu bro? hbu doing today? tysm btw',
        'idc tbh, lowkey was kinda mid ngl',
        'bruh that was wild, im dead, lmao',
        'sus af, big yikes, periodt no cap',
    ],
    'pirate': [
        'Arrr matey, batten down the hatches before the storm hits!',
        'Shiver me timbers, ye scurvy dog, where be me grog?',
        'Avast ye landlubbers, prepare to be boarded by old Blackbeard!',
        'Yo ho ho and a bottle of rum, me hearties, drink up!',
        'X marks the spot, savvy? The treasure lies buried beneath.',
        'Aye captain, the wind be in our favor, lets set sail!',
        'Walk the plank, ye mangy bilge rat, or face me cutlass!',
        'By Davy Jones locker, that be the biggest kraken I ever saw!',
        'Hoist the Jolly Roger, men, theres a galleon on the horizon!',
        'Pieces of eight! Pieces of eight! cried the parrot on me shoulder.',
    ],
    'noir_detective': [
        'The dame walked into my office with trouble written all over her.',
        'Rain was falling like the city was crying for its sins.',
        'I lit another cigarette and watched the smoke curl toward the ceiling.',
        'She had legs that went on forever and a story that didnt add up.',
        'The streets were mean and the dames were meaner that summer.',
        'Whiskey burned its way down my throat, but it didnt warm me.',
        'Some cases stay with you, like the smell of cordite after a shooting.',
        'He had the look of a man whod seen too much and slept too little.',
        'The phone rang at three AM. Bad news always comes after midnight.',
        'I leaned back in my chair and waited for her to spill the whole story.',
    ],
    'gen_z': [
        'Bro that was bussin fr no cap, on god.',
        'Ate and left no crumbs, periodt, slay queen.',
        'It is giving main character energy lowkey.',
        'Sus behavior bestie, im not gonna lie.',
        'Sheeesh, that fit is bussin, no cap on god.',
        'The vibes are immaculate today, im finna eat.',
        'He understood the assignment, mid asf tho.',
        'Rizzler caught the W, certified ohio moment.',
        'Skibidi toilet gyatt, fanum tax, my bro.',
        'Its giving cringe, im gonna touch grass.',
    ],
}
print({k: len(v) for k, v in STYLES.items()})

{'biblical': 10, 'shakespearean': 10, '1920s_slang': 10, 'academic_jargon': 10, 'txt_speak': 10, 'pirate': 10, 'noir_detective': 10, 'gen_z': 10}


In [4]:
style_results = discover(STYLES, NEUTRAL)


=== biblical ===
  feat    92  diff=119.03  cons=0.90  neutral=13.60
  feat 11034  diff=111.70  cons=1.00  neutral=0.00
  feat  2113  diff=84.43  cons=1.00  neutral=0.00
  feat   131  diff=55.67  cons=0.50  neutral=42.20
  feat   331  diff=41.65  cons=0.80  neutral=0.00
  feat  4336  diff=31.26  cons=0.70  neutral=0.00

=== shakespearean ===
  feat    92  diff=158.45  cons=1.00  neutral=13.60
  feat   131  diff=90.24  cons=0.80  neutral=42.20
  feat   494  diff=60.83  cons=1.00  neutral=4.04
  feat  1863  diff=46.94  cons=1.00  neutral=0.00
  feat   291  diff=34.44  cons=0.70  neutral=17.56
  feat  5730  diff=34.38  cons=0.90  neutral=0.00

=== 1920s_slang ===
  feat    92  diff=134.74  cons=1.00  neutral=13.60
  feat   131  diff=110.45  cons=0.90  neutral=42.20
  feat   291  diff=76.07  cons=1.00  neutral=17.56
  feat   494  diff=64.57  cons=0.90  neutral=4.04
  feat   272  diff=36.24  cons=1.00  neutral=10.11
  feat  2213  diff=34.34  cons=0.50  neutral=6.48

=== academic_jargon ===

In [5]:
NP_BASE = f'https://www.neuronpedia.org/gemma-scope-2-1b-it/{LAYER}-gemmascope-res-16k'
for label, rows in style_results.items():
    print(f'\n{label}:')
    for r in rows[:5]:
        print(f'  {NP_BASE}/{r["feat_id"]}  (diff={r["diff_score"]:.2f}, cons={r["consistency"]:.2f})')


biblical:
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/92  (diff=119.03, cons=0.90)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/11034  (diff=111.70, cons=1.00)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/2113  (diff=84.43, cons=1.00)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/131  (diff=55.67, cons=0.50)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/331  (diff=41.65, cons=0.80)

shakespearean:
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/92  (diff=158.45, cons=1.00)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/131  (diff=90.24, cons=0.80)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/494  (diff=60.83, cons=1.00)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/1863  (diff=46.94, cons=1.00)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmas

In [6]:
Path('../features_styles.json').write_text(json.dumps({
    'model': MODEL_ID, 'sae_release': 'gemma-scope-2-1b-it-res', 'sae_id': SAE_ID, 'layer': LAYER,
    'features': style_results,
}, indent=2))
print('wrote features_styles.json')

wrote features_styles.json


## 2. Personas

Distinct voices/character types. Each prompt is a complete utterance in that persona's voice.

In [7]:
PERSONAS = {
    'conspiracy_theorist': [
        'Wake up sheeple, theyre lying to us about everything!',
        'The government is hiding the truth about chemtrails, do your research.',
        'Connect the dots, the elites are controlling the whole narrative.',
        'Big Pharma doesnt want you to know about this one simple trick.',
        'They put it in the water to keep us docile, open your eyes.',
        'The mainstream media is owned by six corporations, think about it.',
        'Crisis actors at every shooting, look at the body language.',
        'They tracked your phone, your car, your fridge, nothing is private.',
        'False flag operation, classic playbook, look at who benefits.',
        'The Illuminati run the world from behind the curtain, always have.',
    ],
    'stoic_philosopher': [
        'The obstacle is the way; embrace what fortune sends.',
        'You have power over your mind, not outside events. Realize this.',
        'It is not death that a man should fear, but never beginning to live.',
        'Waste no more time arguing what a good man should be. Be one.',
        'The happiness of your life depends upon the quality of your thoughts.',
        'Confine yourself to the present moment and accept what is.',
        'Difficulties strengthen the mind, as labor does the body.',
        'When you arise in the morning, think of what a precious privilege it is to be alive.',
        'Memento mori. Remember thou art mortal, and live accordingly.',
        'External things are not the problem; it is your judgment of them.',
    ],
    'scientist': [
        'The data, when properly controlled for confounders, suggests a clear effect.',
        'Hypothesis: increased temperature correlates with enzymatic activity.',
        'We must replicate these findings before drawing firm conclusions.',
        'The p-value indicates the result is unlikely due to chance alone.',
        'Methodologically, the sample size limits our statistical power here.',
        'These observations are consistent with the predicted phase transition.',
        'Falsifiability is the hallmark of a genuine scientific theory.',
        'Peer review is essential to validate the rigor of the experimental design.',
        'Correlation, as always, does not imply causation in this case.',
        'The mechanism appears to involve a regulatory feedback loop.',
    ],
    'valley_girl': [
        'Like, omg, totally, I literally cannot even right now.',
        'That is, like, sooo random, you know what I mean?',
        'Whatever, gag me with a spoon, this is the worst.',
        'Like, hello? Are you even, like, listening to me?',
        'For sure, for sure, totally tubular, I am like obsessed.',
        'Oh my god Becky, look at her outfit, its like so cute.',
        'Totally awesome, like, in the most amazing way ever.',
        'I was like, and then she was like, and then I was like dying.',
        'Bitchin, like, I cant believe he even said that, hello?',
        'As if, like, whatever, I am so over this whole thing.',
    ],
    'corporate_exec': [
        'Lets circle back on this and table the discussion for now.',
        'We need to leverage our synergies for maximum stakeholder value.',
        'Going forward, lets align on the deliverables and timeline.',
        'Im going to need you to take this offline and ping me later.',
        'At the end of the day, its about moving the needle on KPIs.',
        'Lets do a deep dive into the data and ideate on next steps.',
        'I want to put a pin in this and revisit after the all-hands.',
        'We should socialize this initiative across the leadership team.',
        'Touch base on the bandwidth available for Q3 strategic priorities.',
        'Move fast and break things, but lets be intentional about it.',
    ],
    'new_age_guru': [
        'Your aura is vibrating at an extraordinary frequency today, beloved.',
        'Manifest your highest timeline by aligning with universal abundance.',
        'The chakras must be balanced for true energetic harmony to flow.',
        'I sense great healing energy radiating from your sacred space.',
        'Trust the journey, beloved, the universe is conspiring in your favor.',
        'Your soul contract was written long before this incarnation began.',
        'Crystals amplify intention; place rose quartz near your heart space.',
        'Cleanse your energetic field with sage and intentional breathwork.',
        'You are a divine being having a temporary human experience.',
        'The full moon is calling you to release what no longer serves you.',
    ],
    'pessimist': [
        'Whats the point, its all going to fall apart anyway, just like always.',
        'Yeah, sure, that will work out, like everything else hasnt.',
        'I knew it. I knew it would go wrong. Theres no point in trying.',
        'Everything good ends, and usually badly. Why bother.',
        'People let you down. Plans collapse. Thats just how it goes.',
        'Dont get your hopes up, it never works out like you imagine.',
        'Of course it broke. Of course it did. Nothing ever lasts.',
        'I tried. I really tried. And it still ended badly. Typical.',
        'Save your optimism for someone who hasnt seen how this ends.',
        'The world is broken, people are cruel, and hope is a lie.',
    ],
    'noir_narrator': [
        'Some nights, the city talked and you didnt want to hear what it said.',
        'I learned long ago that hope is a fool, and trust is a luxury.',
        'The kind of dame who could make a saint reach for his service revolver.',
        'Everybody in this town wore a face, and nobody wore their own.',
        'You either die a sucker or live long enough to become one.',
        'Truth in this town came wrapped in cigarette smoke and old bourbon.',
        'I was running on fumes and the half-promise of a dying lead.',
        'In my business, you learn to read people the way priests read scripture.',
        'The rain didnt wash the streets clean. Nothing ever does.',
        'She had a smile that could open doors and a past that closed them.',
    ],
}
print({k: len(v) for k, v in PERSONAS.items()})

{'conspiracy_theorist': 10, 'stoic_philosopher': 10, 'scientist': 10, 'valley_girl': 10, 'corporate_exec': 10, 'new_age_guru': 10, 'pessimist': 10, 'noir_narrator': 10}


In [8]:
persona_results = discover(PERSONAS, NEUTRAL)


=== conspiracy_theorist ===
  feat   856  diff=69.34  cons=0.90  neutral=0.00
  feat  5315  diff=44.08  cons=0.80  neutral=0.00
  feat   262  diff=42.64  cons=0.80  neutral=0.00
  feat    92  diff=42.63  cons=0.40  neutral=13.60
  feat   291  diff=29.21  cons=0.50  neutral=17.56
  feat   131  diff=24.79  cons=0.10  neutral=42.20

=== stoic_philosopher ===
  feat    92  diff=74.42  cons=0.70  neutral=13.60
  feat   131  diff=37.24  cons=0.20  neutral=42.20
  feat   701  diff=35.67  cons=0.90  neutral=0.35
  feat  2361  diff=33.63  cons=0.90  neutral=0.00
  feat   249  diff=23.18  cons=0.10  neutral=6.06
  feat   485  diff=22.69  cons=0.40  neutral=21.57

=== scientist ===
  feat  1081  diff=48.37  cons=0.90  neutral=0.00
  feat  1059  diff=32.49  cons=0.50  neutral=0.49
  feat   672  diff=27.88  cons=0.70  neutral=0.39
  feat    52  diff=26.85  cons=0.40  neutral=7.51
  feat  6851  diff=17.41  cons=0.60  neutral=0.00
  feat   268  diff=16.62  cons=0.60  neutral=5.61

=== valley_girl ==

In [9]:
for label, rows in persona_results.items():
    print(f'\n{label}:')
    for r in rows[:5]:
        print(f'  {NP_BASE}/{r["feat_id"]}  (diff={r["diff_score"]:.2f}, cons={r["consistency"]:.2f})')


conspiracy_theorist:
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/856  (diff=69.34, cons=0.90)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/5315  (diff=44.08, cons=0.80)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/262  (diff=42.64, cons=0.80)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/92  (diff=42.63, cons=0.40)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/291  (diff=29.21, cons=0.50)

stoic_philosopher:
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/92  (diff=74.42, cons=0.70)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/131  (diff=37.24, cons=0.20)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/701  (diff=35.67, cons=0.90)
  https://www.neuronpedia.org/gemma-scope-2-1b-it/13-gemmascope-res-16k/2361  (diff=33.63, cons=0.90)
  https://www.neuronpedia.org/gemma-scope-2-1b-it

In [ ]:
Path('../features_personas.json').write_text(json.dumps({
    'model': MODEL_ID, 'sae_release': 'gemma-scope-2-1b-it-res', 'sae_id': SAE_ID, 'layer': LAYER,
    'features': persona_results,
}, indent=2))
print('wrote features_personas.json')

wrote features_personas.json


: 

## Next

- Read top features per style/persona. Look for clean isolated features (high consistency, neutral_act ~0, not appearing in other categories).
- Cross-check on Neuronpedia.
- Add curated picks to `steer.py` STYLES/PERSONAS dicts (next step after this notebook).